# Week 14: An LLM Application Is More Than a Model Call

This notebook follows the reviewed Week 14 presentation. Read each concept and calculate the worked example before running its code.

## Lesson map

1. An LLM Application Is More Than a Model Call
2. Authentication Proves Which Application Is Calling
3. Model Selection Starts with Required Capabilities
4. Streaming Delivers Partial Output Events
5. Tokens Drive Context, Latency, and Cost
6. Retries Handle Transient Failures, Not Every Failure
7. A Provider Adapter Normalizes What Can Be Normalized
8. LangChain Composes Models, Prompts, Parsers, and Tools
9. Tracing Records Every Important Boundary
10. MCP Standardizes Tools and Resources
11. Resilience Needs Explicit Fallback Behaviour
12. Guided Lab: Build a Provider-Resilient Service

Use the same reasoning loop throughout: **predict, run, inspect, explain**.


## 1. An LLM Application Is More Than a Model Call

An engineered LLM application includes:

- request validation;
- prompt and schema versioning;
- provider client;
- model selection;
- retries and timeouts;
- structured-output validation;
- tool authorization;
- tracing and metrics;
- fallback and failure behaviour;
- cost and token controls.

The application owns reliability and security. A provider response is untrusted input until validated.

### Work it out first

User request flows through:

`API -> validation -> prompt -> provider -> schema check -> policy check -> response`

If schema validation fails, the application returns a controlled error or bounded retry rather than storing malformed data.

### Notebook bridge

The LiteLLM and LangChain notebooks expose provider calls that this week wraps in application contracts.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
request = RequestModel.model_validate(payload)
raw = llm_client.generate(build_messages(request))
result = ResponseModel.model_validate(raw)

Expected output:

```text
A typed response or a controlled validation failure.
```


## 2. Authentication Proves Which Application Is Calling

Provider APIs require **authentication**, often through an API key or cloud identity.

Rules:

- keep credentials outside source code and notebooks;
- use environment variables or a secret manager;
- separate development and production credentials;
- grant minimum required permissions and spend limits;
- rotate exposed keys;
- never log authorization headers;
- fail clearly when configuration is missing.

Authentication identifies a caller. **Authorization** decides what that caller may do.

### Work it out first

Unsafe notebook:

`api_key = "live-key-value"`

Safer application:

`api_key = os.environ["PROVIDER_API_KEY"]`

The deployment environment injects the secret; the repository never stores it.

### Notebook bridge

Provider notebooks should read keys from environment variables, never checked-in cells.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
import os

api_key = os.environ["PROVIDER_API_KEY"]
client = ProviderClient(api_key=api_key)

Expected output:

```text
An authenticated client or a clear startup error when the secret is absent.
```


## 3. Model Selection Starts with Required Capabilities

Select a model using:

- task quality on your evaluation set;
- required context length;
- structured-output and tool support;
- latency;
- input and output price;
- data location and retention terms;
- rate limits;
- language and modality support;
- deployment constraints.

Model names and capabilities change. Pin a model identifier where supported and record the observed version.

### Work it out first

Task requires JSON extraction under `2` seconds.

Model A: 98% field accuracy, 4 seconds.  
Model B: 96% field accuracy, 1 second.  
Model C: 90% field accuracy, 0.5 seconds.

Selection depends on the minimum accepted quality and latency requirement.

### Notebook bridge

The LiteLLM notebook sends the same task to several providers for controlled comparison.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
requirements = {
    "structured_output": True,
    "p95_latency_ms": 2000,
    "field_accuracy_min": 0.95,
}

Expected output:

```text
A capability and acceptance contract used to compare candidate models.
```


## 4. Streaming Delivers Partial Output Events

**Streaming** returns partial events while generation continues.

Benefits:

- faster time to first visible token;
- progressive user feedback;
- ability to cancel long generations.

Engineering requirements:

- assemble chunks in order;
- handle disconnects and cancellation;
- avoid treating partial JSON as valid;
- record complete and failed streams;
- apply final validation after completion.

Streaming improves perceived latency, not total generation work.

### Work it out first

Chunks:

`"The "` -> `"answer "` -> `"is 42."`

The interface can display text progressively. For JSON, buffer until the complete object arrives, then parse and validate.

### Notebook bridge

Learners add streaming to one provider call and measure time to first token and total time.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
parts = []
for event in client.stream(messages):
    parts.append(event.text)
complete_text = "".join(parts)

Expected output:

```text
One ordered final string assembled from provider events.
```


## 5. Tokens Drive Context, Latency, and Cost

Provider usage commonly separates:

- input tokens;
- output tokens;
- sometimes cached or reasoning tokens.

Approximate request cost:

`input_tokens x input_rate + output_tokens x output_rate`

Rates must use the provider's current units, often price per million tokens.

Set limits on input size, retrieved context, output length, total request cost, and daily spend.

### Work it out first

Input `2,000` tokens at `$1` per million:

`2000 / 1,000,000 x $1 = $0.002`

Output `500` tokens at `$4` per million:

`500 / 1,000,000 x $4 = $0.002`

Estimated total `$0.004`.

### Notebook bridge

The lab logs token counts, latency, and estimated cost for each provider.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
estimated = input_tokens / 1_000_000 * input_rate
estimated += output_tokens / 1_000_000 * output_rate

Expected output:

```text
A per-request estimate using current provider rates and observed token counts.
```


## 6. Retries Handle Transient Failures, Not Every Failure

Retry only failures likely to be temporary, such as some timeouts, connection failures, `429` rate limits, or provider `5xx` responses.

Do not retry invalid credentials, malformed requests, or permanent policy failures.

Use:

- per-attempt timeout;
- maximum attempts;
- exponential backoff;
- random jitter;
- total time budget;
- idempotency protections for actions.

### Work it out first

Backoff base `1` second:

attempt delays approximately `1`, `2`, and `4` seconds, plus jitter.

With a three-attempt cap and ten-second total budget, the request cannot retry forever.

### Notebook bridge

The provider wrapper adds bounded retries around transient failures.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
for attempt in range(3):
    try:
        return call_provider(timeout=10)
    except TransientError:
        sleep(2**attempt + random.random())
raise ProviderUnavailable()

Expected output:

```text
Success within the retry budget or one controlled unavailable error.
```


## 7. A Provider Adapter Normalizes What Can Be Normalized

A **provider adapter** gives application code one interface for several providers.

LiteLLM can normalize common request and response shapes, token usage, exceptions, routing, and fallbacks.

Capabilities still differ:

- tool and schema formats;
- supported parameters;
- context limits;
- multimodal input;
- safety behaviour;
- streaming events.

The application should check required capabilities instead of assuming every provider behaves identically.

### Work it out first

Application call:

`complete(model_alias, messages, schema)`

Configuration maps `model_alias` to Provider A or B. If Provider B lacks strict schema support, the capability check blocks that route rather than silently weakening validation.

### Notebook bridge

The LiteLLM notebook compares text, structured output, and tool calls across providers.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
from litellm import completion

response = completion(model=model_id, messages=messages)

Expected output:

```text
A normalized completion response containing provider-specific metadata when available.
```


## 8. LangChain Composes Models, Prompts, Parsers, and Tools

LangChain provides interfaces and composition tools for:

- chat models;
- prompt templates;
- structured output;
- retrieval;
- tools;
- streaming;
- tracing.

A framework reduces integration code but does not remove the need to understand message roles, schema validation, retries, permissions, or evaluation.

Prefer small explicit chains whose inputs and outputs are typed and observable.

### Work it out first

Chain:

`validated request -> prompt template -> chat model -> structured output model`

The chain's input is a topic string. Its output is a validated `Summary` object, not unparsed prose.

### Notebook bridge

The LangChain foundations notebook covers model calls, structured output, PDFs, tools, and tracing.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
structured_model = model.with_structured_output(Summary)
chain = prompt | structured_model
result = chain.invoke({"document": document})

Expected output:

```text
A validated Summary object when the provider and schema contract succeed.
```


## 9. Tracing Records Every Important Boundary

A **trace** records one end-to-end application request. A **span** records one operation inside it.

Capture:

- trace and request ID;
- prompt and schema version;
- provider and model;
- token usage and latency;
- retry count;
- tool proposals and results;
- validation outcomes;
- final status and error category;
- cost estimate.

Protect personal data and secrets through redaction and retention rules.

### Work it out first

One request trace contains:

`validate 3ms -> retrieve 80ms -> model 1200ms -> parse 4ms`

Total observed span time is approximately `1,287ms`, revealing that the model call dominates latency.

### Notebook bridge

Learners enable tracing around the LangChain and LiteLLM calls.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
with tracer.start_as_current_span("llm_request") as span:
    span.set_attribute("model", model_id)
    result = call_model()

Expected output:

```text
One searchable trace connecting request metadata and nested operation timings.
```


## 10. MCP Standardizes Tools and Resources

The **Model Context Protocol (MCP)** standardizes how an application connects to servers that expose:

- **tools:** bounded operations the client may request;
- **resources:** readable context with stable identifiers;
- **prompts:** reusable prompt templates where supported.

An MCP client discovers capabilities and sends requests. The server still enforces authentication, authorization, input validation, path boundaries, timeouts, and output limits.

MCP does not make every connected capability safe to expose.

### Work it out first

Filesystem MCP server:

- resource: approved course handout;
- tool: search within approved course directory;
- forbidden: read arbitrary paths or write files.

The server resolves and checks paths before access.

### Notebook bridge

MCP is introduced conceptually before agent workflows use external tools.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
Client -> list tools -> call read_course_file(path)
Server -> validate root and size -> return content or denied error

Expected output:

```text
A bounded resource result whose origin and permission checks are explicit.
```


## 11. Resilience Needs Explicit Fallback Behaviour

A resilient request path defines:

- validation failure;
- context-too-large handling;
- rate-limit and timeout behaviour;
- schema failure;
- provider outage;
- tool denial or timeout;
- fallback eligibility;
- user-visible error;
- trace and alert.

A fallback model is acceptable only if it supports required capabilities and passes the task's quality threshold. Silent fallback can change output quality, data location, or cost.

### Work it out first

Primary provider times out twice.

If the request requires strict JSON and approved fallback supports it, route once to fallback and record the change. Otherwise return a controlled unavailable error.

### Notebook bridge

The LiteLLM lab tests one simulated timeout and one capability-incompatible fallback.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
if primary_failed and fallback.supports(required_capabilities):
    return fallback.generate(request)
raise ServiceUnavailable()

Expected output:

```text
One explicit fallback result or a bounded unavailable response.
```


## 12. Guided Lab: Build a Provider-Resilient Service

Build a small extraction API that:

1. validates request input;
2. reads credentials from environment;
3. supports two provider configurations;
4. checks structured-output capability;
5. streams text only where partial output is safe;
6. limits input and output tokens;
7. records token use, latency, and estimated cost;
8. retries only transient failures;
9. validates every final object;
10. traces model and tool spans;
11. exposes one read-only tool or MCP resource;
12. demonstrates fallback and controlled failure.

### Work it out first

Test matrix:

- valid request;
- invalid schema;
- provider timeout;
- rate limit;
- unsupported capability;
- tool permission denial;
- output over budget.

Each case has an expected status and trace.

### Notebook bridge

Complete `26.multiple-llm-providers-with-litellm.ipynb` and `27.langchain-foundations.ipynb`.


Before running the next cell:

1. Identify every input value.
2. Predict the result or shape.
3. Run the cell.
4. Explain any difference between your prediction and the output.


In [ ]:
result = service.extract(request)
assert isinstance(result, ExtractedRecord)
assert trace.token_usage <= request.token_budget

Expected output:

```text
A validated result within budget, or a typed controlled failure.
```


## Guided lab

Build a small extraction API that:

1. validates request input;
2. reads credentials from environment;
3. supports two provider configurations;
4. checks structured-output capability;
5. streams text only where partial output is safe;
6. limits input and output tokens;
7. records token use, latency, and estimated cost;
8. retries only transient failures;
9. validates every final object;
10. traces model and tool spans;
11. exposes one read-only tool or MCP resource;
12. demonstrates fallback and controlled failure.

### Reference result

Test matrix:

- valid request;
- invalid schema;
- provider timeout;
- rate limit;
- unsupported capability;
- tool permission denial;
- output over budget.

Each case has an expected status and trace.


In [ ]:
# Guided lab workspace: Week 14
# Add only the imports needed for the current step.

# TODO 1: Prepare the smallest valid input.

# TODO 2: Apply the concept taught in this lesson.

# TODO 3: Display inspectable intermediate evidence.

# TODO 4: Compare the result with a hand calculation or stated requirement.

## Weekly deliverable

Submit the completed guided lab with:

- your prediction before execution;
- intermediate values, shapes, metrics, or traces;
- one failed assumption and its correction;
- a plain-English explanation of the result;
- the source notebook section you are now ready to complete.


## Sources and source notebooks

- <https://github.com/curiousily/AI-Bootcamp/blob/master/26.multiple-llm-providers-with-litellm.ipynb>
- <https://docs.langchain.com/oss/python/langchain/overview>
- <https://cheatsheetseries.owasp.org/cheatsheets/Secrets_Management_Cheat_Sheet.html>
- <https://docs.aws.amazon.com/secretsmanager/latest/userguide/best-practices.html>
- <https://docs.litellm.ai/>
- <https://docs.langchain.com/langsmith/evaluation>
- <https://docs.litellm.ai/docs/completion/stream>
- <https://docs.langchain.com/oss/python/langchain/streaming>
- <https://docs.litellm.ai/docs/completion/token_usage>
- <https://huggingface.co/docs/transformers/main_classes/text_generation>
- <https://docs.aws.amazon.com/prescriptive-guidance/latest/cloud-design-patterns/retry-backoff.html>
- <https://docs.litellm.ai/docs/proxy/reliability>
- <https://github.com/curiousily/AI-Bootcamp/blob/master/27.langchain-foundations.ipynb>
- <https://docs.langchain.com/langsmith/observability>
- <https://docs.langchain.com/langsmith/trace-litellm>
- <https://modelcontextprotocol.io/docs/getting-started/intro>
- <https://modelcontextprotocol.io/docs/learn/architecture>